# Bootstrap inicial de Colab

Este notebook se ejecuta **una sola vez por máquina o cuenta nueva**. Su trabajo es dejar todo preparado en Google Drive para que después los notebooks normales (`00_download_datasets.ipynb`, `02_dataset_audit.ipynb`, etc.) arranquen con solo dos líneas.

Lo que va a hacer:
1. Montar Google Drive.
2. Crear en Drive la estructura de carpetas del proyecto (`raw/`, `processed/`, `checkpoints/`, `logs/`, `.ssh/`).
3. Generar (o reutilizar si ya existe) una clave SSH guardada en Drive que servirá para autenticar Colab contra GitHub.
4. Imprimir la clave pública para que la añada como *Deploy Key* en el repositorio (esto solo se hace la primera vez).
5. Clonar el repositorio del proyecto dentro de la VM.
6. Copiar el script `setup/colab_init.sh` a Drive, para que cualquier notebook futuro pueda invocarlo directamente.
7. Instalar las dependencias de Python.
8. Comprobar que PyTorch ve la GPU y que `phmd` se importa.

Antes de empezar: **Runtime → Change runtime type → GPU** (A100 si está disponible).

## 1. Configuración

Constantes que definen dónde está el repositorio, qué rama usamos y cómo se llama la carpeta del proyecto en Drive. Si en algún momento cambia algo (rama, repo, etc.) basta con tocar esta celda.

In [ ]:
REPO_SSH        = "git@github.com:sanzzjose/phm-foundation-model.git"
REPO_BRANCH     = "feature/exploration_phmd"
GIT_USER_NAME   = "Jose Sanz"
GIT_USER_EMAIL  = "josesanzduran@gmail.com"
DRIVE_ROOT_NAME = "fm_fl_phmd"

## 2. Montar Google Drive

Colab no monta Drive automáticamente. Al ejecutar la celda se abrirá una ventana pidiendo permiso para acceder con la cuenta de Google. Una vez aceptado, todo el contenido de Drive queda visible bajo `/content/drive/MyDrive/`.

Aclaración: la API de Colab solo permite montar el Drive completo, no una subcarpeta. Por convención, este proyecto trabaja únicamente con `MyDrive/fm_fl_phmd/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Estructura de carpetas en Drive

Se crean (si no existen) las cinco subcarpetas que usaremos a lo largo del TFM:

- `raw/` — datasets originales descargados de phmd (un único origen de verdad para los datos sin procesar).
- `processed/` — ventanas, normalización y patches generados por el pipeline de armonización.
- `checkpoints/` — snapshots del modelo durante el entrenamiento.
- `logs/` — logs de descarga, entrenamiento, tensorboard, etc.
- `.ssh/` — clave SSH propia de Colab para empujar a GitHub desde la VM.

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive') / DRIVE_ROOT_NAME

subcarpetas = ['raw', 'processed', 'checkpoints', 'logs', '.ssh']

for nombre in subcarpetas:
    ruta = DRIVE_ROOT / nombre
    ruta.mkdir(parents=True, exist_ok=True)
    print(f'  {nombre:12s} {ruta}')

## 4. Clave SSH para que Colab pueda hablar con GitHub

Necesitamos que la VM de Colab pueda hacer `git clone` y `git push` contra el repositorio. La forma estándar y segura es usar una clave SSH.

Estrategia: generar la clave **una sola vez** y guardarla en Drive (`MyDrive/fm_fl_phmd/.ssh/`). En las sesiones siguientes, en lugar de regenerarla, simplemente la copiamos de Drive a `~/.ssh` de la VM. Así la misma clave persiste entre sesiones, aunque la VM cambie.

**Solo la primera vez:** al ejecutar la celda se imprime la *clave pública*. Hay que copiarla y añadirla como **Deploy Key con permiso de escritura** en:
https://github.com/sanzzjose/phm-foundation-model/settings/keys/new

In [ ]:
import os
import subprocess
from pathlib import Path

# La clave vive en Drive para que sobreviva al reinicio de la VM
clave_priv = DRIVE_ROOT / '.ssh' / 'id_ed25519_colab'
clave_pub  = DRIVE_ROOT / '.ssh' / 'id_ed25519_colab.pub'

# Si todavía no existe, la generamos. ssh-keygen con -N '' deja la clave sin passphrase
# (cómodo dentro de Colab, donde no hay un agente para almacenarla)
if not clave_priv.exists():
    subprocess.run(
        ['ssh-keygen', '-t', 'ed25519', '-C', 'colab@fm_fl_phmd',
         '-f', str(clave_priv), '-N', ''],
        check=True,
    )
    print('Clave SSH nueva creada en Drive')
else:
    print('Ya existía una clave SSH en Drive, la reutilizamos')

# Copiamos la clave a ~/.ssh de la VM (es donde ssh la busca por defecto)
vm_ssh = Path.home() / '.ssh'
vm_ssh.mkdir(mode=0o700, exist_ok=True)
(vm_ssh / 'id_ed25519').write_text(clave_priv.read_text())
(vm_ssh / 'id_ed25519.pub').write_text(clave_pub.read_text())
os.chmod(vm_ssh / 'id_ed25519',     0o600)  # la privada solo el dueño puede leer
os.chmod(vm_ssh / 'id_ed25519.pub', 0o644)

# Confiamos en github.com para que el primer ssh no pregunte interactivamente
subprocess.run(
    'ssh-keyscan -t ed25519,rsa github.com >> ~/.ssh/known_hosts 2>/dev/null',
    shell=True, check=False,
)

# Imprimimos la pubkey para añadirla a GitHub como Deploy Key (solo la primera vez)
print()
print('=' * 70)
print('Clave pública (añadir como Deploy Key con write access en GitHub):')
print('=' * 70)
print(clave_pub.read_text())
print('Enlace: https://github.com/sanzzjose/phm-foundation-model/settings/keys/new')

## 5. Clonar el repositorio

Si es la primera vez que ejecutas este notebook, antes de seguir asegúrate de haber añadido la pubkey de la celda anterior como Deploy Key en GitHub. Si no, el clone fallará por falta de autenticación.

Convención de rutas: el repositorio vive en `/content/fm_fl_phmd` dentro de la VM (no en Drive — sería muy lento). Cada sesión nueva clona o actualiza ahí.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

repo_local = Path('/content/fm_fl_phmd')

# Identidad para los commits que hagamos desde la VM
subprocess.run(['git', 'config', '--global', 'user.email', GIT_USER_EMAIL], check=True)
subprocess.run(['git', 'config', '--global', 'user.name',  GIT_USER_NAME ], check=True)

# Salimos de la carpeta destino antes de tocarla, para evitar errores de cwd
os.chdir('/content')

if (repo_local / '.git').exists():
    # Ya está clonado de una sesión anterior: actualizamos
    subprocess.run(['git', '-C', str(repo_local), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(repo_local), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(repo_local), 'pull', '--ff-only'], check=True)
else:
    # Si hay restos de un clone roto, los limpiamos antes de volver a clonar
    if repo_local.exists():
        shutil.rmtree(repo_local)
    subprocess.run(['git', 'clone', '-b', REPO_BRANCH, REPO_SSH, str(repo_local)], check=True)

subprocess.run(['git', '-C', str(repo_local), 'status'], check=False)

## 6. Copiar `colab_init.sh` a Drive

El script `setup/colab_init.sh` es el que usarán todos los demás notebooks como única celda de arranque. Lo copiamos a Drive para que esté accesible incluso antes de tener el repo clonado en una sesión nueva.

A partir de aquí, cualquier notebook puede arrancar con solo dos líneas:

```python
from google.colab import drive; drive.mount('/content/drive')
!bash /content/drive/MyDrive/fm_fl_phmd/colab_init.sh
```

In [ ]:
import shutil
from pathlib import Path

origen = Path('/content/fm_fl_phmd/setup/colab_init.sh')
destino = DRIVE_ROOT / 'colab_init.sh'

shutil.copy(origen, destino)
destino.chmod(0o755)  # ejecutable

print(f'Copiado {origen} -> {destino}')

## 7. Dependencias

Instalamos los paquetes Python que necesita el proyecto. Están listados en `setup/requirements.txt` para tenerlo todo centralizado y poder revisar/actualizar versiones de un solo sitio.

Después de esta celda, Colab puede emitir un aviso pidiendo reiniciar el runtime. Si lo pide, hazlo (Runtime → Restart session) y vuelve a ejecutar a partir de la celda 8.

In [ ]:
!pip install -q -r /content/fm_fl_phmd/setup/requirements.txt

## 8. Comprobaciones rápidas

Verificamos que la GPU está visible para PyTorch y que `phmd` se importa sin errores. Si alguno falla, lo más probable es que falte reiniciar la runtime tras la instalación.

In [ ]:
import torch
import phmd

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA OK  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
print('phmd     : importado correctamente')

## Listo

A partir de aquí no hace falta volver a ejecutar este notebook salvo que cambie algo del setup (clave, repo, etc.). Para trabajar abre el notebook que toque (`00_download_datasets.ipynb`, `02_dataset_audit.ipynb`, …) y deja que su primera celda llame a `colab_init.sh`.